# MAANG Stocks: Analysis

This notebook downloads one year of daily adjusted close prices for MAANG stocks (Meta, Apple, Amazon, Netflix, Google). It 

- computes daily simple returns and log returns, 
- summarizes their distributional properties, 
- compares empirical distributions to a normal approximation using sample mean and standard deviation, 
- evaluates serial dependence using autocorrelation and Ljung-Box tests.

In [ ]:
import datetime
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, jarque_bera, norm
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

# Define the MAANG tickers for analysis
tickers = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL']

# Download one year of daily adjusted close prices
end_date = datetime.date.today()
start_date = end_date - datetime.timedelta(days=365)
prices = yf.download(tickers, start=start_date, end=end_date, progress=False, actions=False)["Close"]
prices = prices.dropna(how='all')

In [ ]:
# Compute daily simple returns and log returns
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

## Distributional Properties

- `data.std(ddof=1)`: unbiased sample standard deviation with Bessel's correction (degrees of freedom = $n-1$):
    $$
    s = \sqrt{\frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2}\,.
    $$
    See https://pandas.pydata.org/docs/reference/api/pandas.Series.std.html
    
- `skew(data, bias=False)`: unbiased sample skewness. The adjusted estimator is:
    $$
    G_1 = \frac{\sqrt{n(n-1)}}{n-2} \frac{m_3}{m_2^{3/2}}\,,
    $$
    where $m_k = \frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^k$ is the k-th central moment and $n$ is the sample size. See https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html

- `kurtosis(data, fisher=True, bias=False)`: unbiased excess kurtosis. The adjusted estimator is:
    $$
    G_2 = \frac{(n-1)}{(n-2)(n-3)}\left((n+1)g_2 + 6\right)\,,
    $$
    where $g_2 = \frac{m_4}{m_2^2} - 3$ is the biased excess kurtosis estimator, and $m_k$ is the k-th central moment. See https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kurtosis.html

- `jarque_bera(data)`: Jarque-Bera test statistic for normality: 
    $$
    \mathrm{JB} = \frac{n}{6} \left(S^2 + \frac{(K-3)^2}{4} \right)\,,
    $$
    where \(S\) is skewness and \(K\) is kurtosis. Returns `(JB, p-value)`. See https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.jarque_bera.html

In [ ]:
pd.set_option("display.multi_sparse", True)


summary = []
for label, series in [('Returns', returns), ('Log Returns', log_returns)]:
    for ticker in tickers:
        data = series[ticker]
        stat = {
            'Ticker': ticker,
            'Type': label,
            'Mean': data.mean(),
            'Std Dev': data.std(ddof=1),
            'Skewness': skew(data, bias=False),
            'Kurtosis': kurtosis(data, fisher=True, bias=False),
        }
        jb_stat, jb_pvalue = jarque_bera(data)
        stat['Jarque-Bera'] = jb_stat
        stat['JB p-value'] = jb_pvalue
        summary.append(stat)

summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values(by=['Ticker', 'Type'])
summary_df = summary_df.round(6)
summary_df

The table above summarizes the first- and second-moment statistics, skewness, excess kurtosis, and Jarque-Bera normality test results for both simple and log returns. The Jarque-Bera test is especially useful for detecting departures from the normal distribution due to skewness and kurtosis.

## Empirical Distribution with Normal Overlay

Plot the empirical distribution of log returns for each ticker and overlay a normal density using the sample mean and standard deviation. This helps visualize departures from normality.

In [ ]:
fig, axes = plt.subplots(len(tickers)+1, 1, figsize=(4, 4*len(tickers)))
axes = axes.flatten()

for idx, ticker in enumerate(tickers):
    data = log_returns[ticker].dropna()
    mu = data.mean()
    sigma = data.std(ddof=1)
    x = np.linspace(data.min(), data.max(), 200)
    ax = axes[idx]
    ax.hist(data, bins=30, density=True, alpha=0.6, color=f'C{idx}', edgecolor='black')
    ax.plot(x, norm.pdf(x, loc=mu, scale=sigma), color='black', linewidth=2, label='Normal approx')
    ax.axvline(mu, color='gray', linestyle='--', linewidth=1)
    ax.set_title(f'{ticker} Empirical Distribution')
    ax.set_xlabel('Log Return')
    ax.set_ylabel('Density')
    ax.legend()

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## Autocorrelation Analysis

Compute the autocorrelation function (ACF) of log returns and then assess serial dependence with the Ljung-Box test.

In [ ]:
nlags = 5
acf_results = {}
for ticker in tickers:
    acf_results[ticker] = acf(log_returns[ticker], nlags=nlags, fft=False)

acf_df = pd.DataFrame(acf_results, index=[f'Lag {i}' for i in range(nlags + 1)])
acf_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for idx, ticker in enumerate(tickers):
    ax.plot(range(nlags + 1), acf_results[ticker], marker='o', label=ticker, linewidth=2)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('ACF of Log Returns for MAANG Stocks')
ax.set_xticks(range(nlags + 1))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Ljung-Box Test for Serial Correlation

The Ljung-Box test checks whether there is statistically significant autocorrelation across multiple lags simultaneously. A p-value below 0.05 suggests evidence against the null hypothesis of no autocorrelation.

In [ ]:
results = []
for ticker in tickers:
    lb_test = acorr_ljungbox(log_returns[ticker], lags=nlags, return_df=True)
    lb_test['Ticker'] = ticker
    results.append(lb_test)

lb_df = pd.concat(results, ignore_index=False)
lb_df = lb_df.reset_index().rename(columns={'index': 'Lag'})
lb_df['Lag'] = lb_df['Lag'] + 1
lb_df = lb_df[['Ticker', 'Lag', 'lb_stat', 'lb_pvalue']].rename(columns={
    'lb_stat': 'LB Statistic',
    'lb_pvalue': 'p-value'
})
lb_df

## Ljung-Box Interpretation

If the Ljung-Box test fails to reject the null hypothesis, the log returns are consistent with a white noise process. A white noise series is defined as a sequence of random variables with constant mean, constant variance, and zero autocorrelation at all non-zero lags.

In this notebook, the Ljung-Box results do not provide strong evidence of predictive lagged structure in the MAANG log returns, supporting the idea that the series behaves like white noise over this sample period.